# 03. 하네스 어닐링과 Experience 진화

목표: 학습이 진행될수록 하네스 호출을 선택적으로 줄이고, 제한된 Experience 저장소에서 유용한 기술을 남기는 과정을 모의 실험합니다. 실제 GRPO 학습이 아니라 논문에서 관찰한 두 동역학을 분리한 결정론적 예제입니다.

In [ ]:
from dataclasses import dataclass
from math import exp
from typing import Dict

@dataclass
class Skill:
    text: str
    uses: int = 1
    confidence: float = 0.5

class ExperienceStore:
    def __init__(self, capacity: int = 4):
        self.capacity = capacity
        self.skills: Dict[str, Skill] = {}

    def upsert(self, key: str, text: str, evidence: float):
        if key in self.skills:
            skill = self.skills[key]
            skill.text = text  # 최신 관찰로 오래된 내용을 수정합니다.
            skill.uses += 1
            skill.confidence = min(1.0, (skill.confidence + evidence) / 2)
        else:
            self.skills[key] = Skill(text=text, confidence=evidence)
        self._evict_lfu()

    def recall(self, key: str):
        skill = self.skills.get(key)
        if skill:
            skill.uses += 1
        return skill

    def _evict_lfu(self):
        while len(self.skills) > self.capacity:
            victim = min(self.skills, key=lambda k: (self.skills[k].uses, self.skills[k].confidence))
            del self.skills[victim]


## 경험 통합과 잘못된 기억 수정

저장소는 무한 누적하지 않고 같은 키를 갱신하며, 용량을 넘으면 사용 빈도와 신뢰도가 낮은 항목을 제거합니다. 현재 관찰이 오래된 기억과 충돌하면 현재 증거로 수정합니다.

In [ ]:
store = ExperienceStore(capacity=4)
events = [
    ('kettle', '주전자는 식탁 근처', 0.45),
    ('soap', '비누는 싱크대 근처', 0.80),
    ('clean', '씻기 전에 물체를 잡는다', 0.90),
    ('heat', '가열 전에 전자레인지를 연다', 0.75),
    ('kettle', '주전자는 조리대에서 먼저 탐색', 0.95),
    ('cool', '냉각에는 냉장고를 사용', 0.85),
]

sizes = []
for key, text, evidence in events:
    store.upsert(key, text, evidence)
    sizes.append(len(store.skills))
    print(f'{key:7} -> {list(store.skills)}')

assert len(store.skills) <= store.capacity
assert store.skills['kettle'].text == '주전자는 조리대에서 먼저 탐색'
print('최종 기술:', {k: (v.text, v.uses, round(v.confidence, 2)) for k, v in store.skills.items()})


## 선택적 호출로의 어닐링

초기에는 탐색을 위해 기본 호출 확률이 높고, 에포크가 지날수록 반복 패턴을 내재화했다고 가정해 감소시킵니다. 다만 불확실성이 높은 과제에는 최소 호출 확률을 남겨 둡니다.

In [ ]:
def harness_probability(epoch: int, uncertainty: float) -> float:
    exploration = 0.85 * exp(-epoch / 35)
    selective_floor = 0.10 + 0.55 * uncertainty
    return min(1.0, max(selective_floor, exploration))

epochs = [0, 10, 30, 60, 100, 150]
for uncertainty in (0.2, 0.8):
    curve = [round(harness_probability(epoch, uncertainty), 3) for epoch in epochs]
    print(f'불확실성={uncertainty}:', dict(zip(epochs, curve)))

early = harness_probability(0, 0.2)
late_easy = harness_probability(150, 0.2)
late_hard = harness_probability(150, 0.8)
assert early > late_easy
assert late_hard > late_easy
print('결론: 전체 호출은 줄지만 어려운 과제에는 선택적 외부 상태 접근이 남습니다.')


## 확장 과제

1. `Skill`에 출처와 만료 시각을 추가하고 오래된 항목을 제거하세요.
2. 현재 관찰과 기억이 충돌할 때 증거 신뢰도에 따라 업데이트 또는 보류하도록 바꾸세요.
3. 호출 확률 대신 성공·비용 보상을 최대화하는 밴딧 정책을 구현하세요.
4. 저장소에 악성 지시가 들어오는 상황을 만들고 스키마 검증과 격리 영역을 추가하세요.